# Renovaí — Carga de Volume (dados de teste)

Este notebook popula o banco **Renovaí** (PostgreSQL) com uma carga de dados
**verossímil** para testes de volume: milhares de registros distribuídos por
todas as entidades do schema, respeitando chaves estrangeiras, `CHECK`s,
triggers de validação e as regras de negócio do domínio.

## O que este notebook assume

- O schema já foi criado (`schema/tables.sql`, `schema/auditoria_logs.sql`,
  `schema/foreign_keys.sql`), assim como funções, procedures, triggers e views
  (`programmability/**`) e o seed de referência
  (`data/seed_dados_referencia.sql`) — ou seja, o `init.sql` do projeto já
  rodou. Este notebook faz a **carga de volume**, não a criação do schema.
- Existe um arquivo `.env` (veja `.env.example`) com as credenciais de conexão.

## Regras de negócio importantes que este notebook respeita

1. **Tabelas que só recebem dados por meio de `CALL` de procedure** (nunca
   `INSERT` direto), pois é assim que o sistema funciona na prática:
   - `triagens` e `movimentacoes_estoques` (entrada) → `sp_registrar_movimentacao_triagem`
   - `movimentacoes_estoques` (saída, aceite direto) → `sp_aceitar_pedido_cooperativa`
   - `movimentacoes_estoques` (saída, negociação) → `sp_fechar_negociacao`
   - `rateios` e `rateios_funcionarios` → `sp_fechar_rateio_mensal`

   Todas as demais tabelas recebem `INSERT` direto em massa
   (`INSERT INTO tabela (...) VALUES (...), (...), ...`).

2. **Pedidos nunca são direcionados a uma cooperativa sem estoque suficiente
   do material pedido.** Cada cooperativa recebe uma carga base de ~5.000 kg
   por material genérico (com variação via `numpy`), e a cada pedido o
   notebook filtra, em Python, apenas as cooperativas cujo estoque comporta
   a quantidade pedida — o mesmo saldo é reconferido pelas próprias
   procedures (`fn_verificar_disponibilidade_estoque`) no banco. Pedidos com
   itens de quantidade muito alta, que nenhuma cooperativa consegue atender,
   são deliberadamente deixados **sem** vínculo em `pedidos_cooperativas`
   (cenário realista: pedido aberto aguardando capacidade).

3. **Relacionamentos 1:1, 1:N e N:N são respeitados:**
   - 1:1 — `perfis` ↔ `empresas` OU `cooperativas` (nunca os dois nem nenhum,
     conforme `CHECK`); `telefones` ↔ `perfis` OU `usuarios`.
   - 1:N — `cooperativas` → `funcionarios`, `pedidos` → `pedido_itens`,
     `cooperativas` → `estoques`, etc.
   - N:N (via tabela associativa) — `pedidos` ↔ `cooperativas` (por
     `pedidos_cooperativas`), `empresas` ↔ `categorias_materiais` (por
     `empresa_materiais_interesses`), `empresas` ↔ `cooperativas` (por
     `empresa_cooperativas_favoritas`), `equipes` ↔ `funcionarios` cooperados
     (por `equipes_cooperados`).

4. A **conexão usa `autocommit=True`**. Isso é obrigatório: as procedures
   (`sp_aceitar_pedido_cooperativa`, `sp_fechar_negociacao`,
   `sp_registrar_movimentacao_triagem`, `sp_fechar_rateio_mensal`) têm um
   `COMMIT` dentro do próprio corpo — o PostgreSQL só permite isso quando a
   chamada `CALL` não está dentro de um bloco de transação aberto por fora.

> **Dica:** rode este notebook contra um banco de teste/homologação antes de
> qualquer ambiente compartilhado — a carga é grande (milhares de linhas) e
> não é totalmente reversível em bloco (cada `INSERT`/`CALL` já é
> commitado).


## 1. Instalação de dependências (rode uma vez, se necessário)

In [1]:
# Descomente se estas bibliotecas ainda não estiverem instaladas no seu ambiente
# %pip install psycopg2-binary Faker numpy python-dotenv
# %pip install bcrypt

## 2. Imports

In [2]:
import datetime
import bcrypt
import os
import random
import uuid

import numpy as np
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv
from faker import Faker

def gerar_token(senha: str = "senha123"):
    return bcrypt.hashpw(
        senha.encode("utf-8"),
        bcrypt.gensalt()
    ).decode("utf-8")

# sementes fixas => carga reprodutível (remova/ajuste se quiser dados diferentes a cada execução)
RNG_SEED = 20260914
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

fake = Faker("pt_BR")
Faker.seed(RNG_SEED)

today = datetime.date.today()
print("Data de referência de hoje:", today)


Data de referência de hoje: 2026-09-21


## 3. Conexão com o banco via `.env`

Crie um arquivo `.env` (veja `.env.example` ao lado) com:

```
DB_HOST=seu-host.neon.tech
DB_PORT=5432
DB_NAME=neondb
DB_USER=seu_usuario
DB_PASSWORD=sua_senha
DB_SSLMODE=require
```


In [3]:
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST", "localhost"),
    port=os.getenv("DB_PORT", "5432"),
    dbname=os.getenv("DB_NAME", "postgres"),
    user=os.getenv("DB_USER", "postgres"),
    password=os.getenv("DB_PASSWORD", ""),
    sslmode=os.getenv("DB_SSLMODE", "require"),
)
# OBRIGATÓRIO: as procedures fazem COMMIT internamente; isso só é permitido
# quando a chamada CALL não está dentro de uma transação aberta pelo driver.
conn.autocommit = True
psycopg2.extras.register_uuid()

cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone()[0])


PostgreSQL 18.6 (6569466) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## 4. Funções auxiliares de carga

- `bulk_insert`: gera **um único `INSERT INTO tabela (...) VALUES (...), (...), ...`**
  por lote (multi-linha, como pedido), usando `execute_values` (parametrizado,
  seguro contra injeção de SQL).
- `call_proc`: executa `CALL procedure(...)`.
- `update_one`: `UPDATE` pontual (usado apenas para sincronizar campos que a
  própria aplicação atualizaria depois de uma procedure, como
  `pedidos.data_conclusao`).


In [4]:
def new_id():
    return uuid.uuid4()


def bulk_insert(cur, table, columns, rows, page_size=500, on_conflict=None):
    """INSERT INTO table (...) VALUES (...), (...), ... em lote.

    `on_conflict`, quando informado, é anexado literalmente após o VALUES
    (ex.: "ON CONFLICT (cooperativa_id, material_id) DO NOTHING") — usado
    nas tabelas com UNIQUE onde a carga precisa ser segura para reexecução
    (rodar o notebook de novo no mesmo banco não deve gerar duplicidade nem
    quebrar com unique_violation).
    """
    if not rows:
        return
    cols_sql = ", ".join(columns)
    template = "(" + ", ".join(["%s"] * len(columns)) + ")"
    values = [tuple(row.get(c) for c in columns) for row in rows]
    sql = f"INSERT INTO {table} ({cols_sql}) VALUES %s"
    if on_conflict:
        sql += f" {on_conflict}"
    psycopg2.extras.execute_values(cur, sql, values, template=template, page_size=page_size)


def call_proc(cur, proc_name, params):
    placeholders = ", ".join(["%s"] * len(params))
    cur.execute(f"CALL {proc_name}({placeholders})", params)


def update_one(cur, table, set_dict, where_col, where_val):
    set_sql = ", ".join(f"{k} = %s" for k in set_dict)
    params = list(set_dict.values()) + [where_val]
    cur.execute(f"UPDATE {table} SET {set_sql} WHERE {where_col} = %s", params)


def uniq_ddd():
    return str(random.choice([11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 31, 41, 51, 61, 71, 81, 85]))


def uniq_telefone_local():
    return f"9{random.randint(1000,9999)}-{random.randint(1000,9999)}"


## 5. Dados de referência já existentes

`cargos`, `status`, `tipos_rateios` e as 4 categorias/materiais macro já
foram inseridos por `data/seed_dados_referencia.sql`. Este notebook **busca**
esses IDs em vez de duplicá-los.


In [5]:
cur.execute("SELECT cargo_id, cargo FROM cargos")
cargos = {cargo: cid for cid, cargo in cur.fetchall()}

cur.execute("SELECT status_id, referencia, status_atual FROM status")
status = {(ref, atual): sid for sid, ref, atual in cur.fetchall()}

cur.execute("SELECT categoria_id, nome_categoria FROM categorias_materiais WHERE categoria_pai_id IS NULL")
categoria_by_nome = {nome: cid for cid, nome in cur.fetchall()}

cur.execute("SELECT material_id, categoria_id FROM materiais WHERE cooperativa_id IS NULL")
materiais_existentes = cur.fetchall()

assert set(cargos) >= {"Cooperado", "Gestor", "Admin", "motorista"}, \
    "Rode data/seed_dados_referencia.sql antes deste notebook (cargos ausentes)."
assert len(status) >= 14, "Rode data/seed_dados_referencia.sql antes deste notebook (status ausentes)."

print("cargos:", cargos)
print("status carregados:", len(status))
print("categorias macro existentes:", categoria_by_nome)
print("materiais genéricos existentes:", len(materiais_existentes))


cargos: {'Cooperado': UUID('2b8aaf66-0b3d-4e4d-940e-3442431a5fb1'), 'Gestor': UUID('59b407f1-0339-4a35-bc6a-4a3bf6083177'), 'Admin': UUID('79437b7a-d758-43bb-b674-306fd7a5d6ab'), 'motorista': UUID('1baf9eac-ae2d-4e95-a386-e070499538e2'), 'ADMIN_SITE': UUID('cd1114d3-5f88-4c8f-a788-0efa3b43e089')}
status carregados: 14
categorias macro existentes: {'Vidro': UUID('c0e3d4ee-91fe-42aa-9680-2edd0285826a'), 'Metal': UUID('f1806c71-51bb-42b7-81bb-4d12647836c4'), 'Plástico': UUID('5373ee9a-5c56-4c5d-88c2-9e412180a4bf'), 'Papel': UUID('84c5bbdd-ee9c-432c-9b16-5e1f77ba20eb')}
materiais genéricos existentes: 4


## 6. Configuração de volume

Ajuste estas constantes para controlar o tamanho da carga. Os valores
padrão já geram **milhares** de registros no total (bem acima do mínimo de
1.000 pedido pelo enunciado).


In [6]:
N_COOPERATIVAS = 18
N_EMPRESAS = 45
N_ENDERECOS = 500
N_USUARIOS_POR_COOP = 22        # 1 Gestor, 2 Admin, 2 motorista, resto Cooperado
N_ADMINS_PLATAFORMA = 4         # funcionários sem cooperativa (admins da plataforma)
N_MATERIAIS_GENERICOS = 60      # cooperativa_id IS NULL -> podem ser pedidos por empresas
N_MATERIAIS_ESPECIFICOS = 10    # cooperativa_id definido -> exclusivos de 1 cooperativa
N_PEDIDOS = 260
N_COLETAS = 450

ESTOQUE_BASE_KG = 5000.0        # carga base pedida no enunciado

emails_usados = set()
cpfs_usados = set()
cnpjs_usados = set()


## 7. Fase 1 — Cooperativas e Empresas

Entidades independentes (sem FK de entrada). `cooperativas.numero_cooperados`
é mantido automaticamente pela trigger `trg_sincronizar_numero_cooperados`
assim que os `funcionarios` forem inseridos (fase 5) — por isso não é
definido aqui.


In [7]:
cooperativas = []
for i in range(N_COOPERATIVAS):
    cooperativas.append({
        "cooperativa_id": new_id(),
        "nome": f"Cooperativa {fake.word().capitalize()} de Reciclagem de {fake.city()}",
        "imagem_url": None,
        "descricao": fake.text(180),
        "contato_preferencial": random.choice(["WHATSAPP", "TELEFONE", "EMAIL"]),
        "horario_funcionamento": "Seg a Sex, 08h-17h",
    })
bulk_insert(cur, "cooperativas",
            ["cooperativa_id", "nome", "imagem_url", "descricao", "contato_preferencial",
             "horario_funcionamento"], cooperativas)

empresas = []
for i in range(N_EMPRESAS):
    empresas.append({
        "empresa_id": new_id(),
        "nome": fake.company(),
        "imagem_url": None,
        "descricao": fake.text(180),
    })
bulk_insert(cur, "empresas", ["empresa_id", "nome", "imagem_url", "descricao"], empresas)

print("cooperativas:", len(cooperativas), "| empresas:", len(empresas))


cooperativas: 18 | empresas: 45


## 8. Fase 2 — Endereços

Independente; usado depois por `perfis` (comerciais) e `rotas_enderecos`
(pontos de coleta).


In [8]:
enderecos = []
tipos_pool = ["Comercial"] * 4 + ["Residencial"] * 3 + ["Coleta"] * 3
for i in range(N_ENDERECOS):
    enderecos.append({
        "endereco_id": new_id(),
        "cep": fake.postcode(),
        "logradouro": fake.street_name(),
        "numero": str(fake.building_number()),
        "complemento": None if random.random() < 0.6 else f"Sala {random.randint(1, 50)}",
        "bairro": fake.bairro(),
        "cidade": fake.city(),
        "tipo": random.choice(tipos_pool),
    })
bulk_insert(cur, "enderecos",
            ["endereco_id", "cep", "logradouro", "numero", "complemento", "bairro", "cidade", "tipo"],
            enderecos)

enderecos_comerciais = [e for e in enderecos if e["tipo"] == "Comercial"]
enderecos_coleta = [e for e in enderecos if e["tipo"] == "Coleta"]
print("enderecos:", len(enderecos), "| comerciais:", len(enderecos_comerciais),
      "| coleta:", len(enderecos_coleta))


enderecos: 500 | comerciais: 191 | coleta: 158


## 9. Fase 3 — Perfis (1:1 com Empresas OU Cooperativas)

`perfis` tem um `CHECK` garantindo que **exatamente um** de `empresa_id` /
`cooperativa_id` esteja preenchido — é o relacionamento 1:1 do domínio
(cada empresa e cada cooperativa tem exatamente um perfil de acesso/login).


In [9]:
perfis = []
perfil_by_cooperativa = {}
perfil_by_empresa = {}

for coop in cooperativas:
    pid = new_id()
    perfis.append({
        "perfil_id": pid, "empresa_id": None, "cooperativa_id": coop["cooperativa_id"],
        "endereco_id": random.choice(enderecos_comerciais)["endereco_id"],
        "email": fake.unique.email(), "cnpj": fake.unique.cnpj(), "senha_hash": None
    })
    perfil_by_cooperativa[coop["cooperativa_id"]] = pid

for emp in empresas:
    pid = new_id()
    perfis.append({
        "perfil_id": pid, "empresa_id": emp["empresa_id"], "cooperativa_id": None,
        "endereco_id": random.choice(enderecos_comerciais)["endereco_id"],
        "email": fake.unique.email(), "cnpj": fake.unique.cnpj(), "senha_hash": gerar_token()
    })
    perfil_by_empresa[emp["empresa_id"]] = pid

bulk_insert(cur, "perfis",
            ["perfil_id", "empresa_id", "cooperativa_id", "endereco_id", "email", "cnpj", "senha_hash"], perfis)
print("perfis:", len(perfis))


perfis: 63


## 10. Fase 4 — Categorias de materiais (subcategorias) e Materiais

O seed de referência já criou 4 categorias macro (Plástico, Papel, Metal,
Vidro) e 1 material genérico por categoria. Aqui:

- criamos **subcategorias** (relação 1:N autorreferenciada via
  `categoria_pai_id`) e mais 2 categorias macro novas (Eletrônicos, Têxtil);
- criamos **materiais genéricos** (`cooperativa_id IS NULL`, podem ser
  pedidos por qualquer empresa/cooperativa) e alguns **materiais
  específicos** de uma cooperativa (`cooperativa_id` definido) — estes,
  pela trigger `fn_validar_material_item`, **não podem** aparecer em
  `pedido_itens` de empresas, apenas em `estoques`/`triagens` da própria
  cooperativa.


In [10]:
novas_macro = {"Eletrônicos": None, "Têxtil": None}
categorias_novas_rows = []
for nome in novas_macro:
    cid = new_id()
    novas_macro[nome] = cid
    categorias_novas_rows.append({"categoria_id": cid, "categoria_pai_id": None, "nome_categoria": nome})

todas_macro = dict(categoria_by_nome)
todas_macro.update(novas_macro)

subcats_por_macro = {
    "Plástico": ["PET", "PEAD", "PVC", "Filme Plástico"],
    "Papel": ["Papelão", "Jornal", "Papel Branco", "Papel Misto"],
    "Metal": ["Alumínio", "Cobre", "Ferro/Aço", "Latinha"],
    "Vidro": ["Vidro Incolor", "Vidro Colorido"],
    "Eletrônicos": ["Placas Eletrônicas", "Cabos", "Baterias"],
    "Têxtil": ["Algodão", "Roupas Usadas"],
}

leaf_categorias = []  # (categoria_id, nome_subcategoria, nome_macro)
for macro_nome, subs in subcats_por_macro.items():
    macro_id = todas_macro[macro_nome]
    for sub in subs:
        cid = new_id()
        categorias_novas_rows.append({"categoria_id": cid, "categoria_pai_id": macro_id,
                                       "nome_categoria": sub})
        leaf_categorias.append((cid, sub, macro_nome))

bulk_insert(cur, "categorias_materiais", ["categoria_id", "categoria_pai_id", "nome_categoria"],
            categorias_novas_rows)

PRECO_RANGE = {
    "Plástico": (1.0, 4.0), "Papel": (0.4, 2.0), "Metal": (2.0, 7.0),
    "Vidro": (1.0, 3.0), "Eletrônicos": (3.0, 12.0), "Têxtil": (0.3, 2.0),
}

materiais_genericos = []
for _ in range(N_MATERIAIS_GENERICOS):
    cat_id, sub_nome, macro_nome = random.choice(leaf_categorias)
    lo, hi = PRECO_RANGE[macro_nome]
    preco = round(float(np.random.uniform(lo, hi)), 2)
    materiais_genericos.append({
        "material_id": new_id(), "categoria_id": cat_id, "cooperativa_id": None,
        "imagem_url": None, "preco_sugerido": preco, "esta_disponivel": random.random() < 0.95,
    })

materiais_especificos = []
coops_com_material_especifico = random.sample(cooperativas, N_MATERIAIS_ESPECIFICOS)
for coop in coops_com_material_especifico:
    cat_id, sub_nome, macro_nome = random.choice(leaf_categorias)
    lo, hi = PRECO_RANGE[macro_nome]
    preco = round(float(np.random.uniform(lo, hi)), 2)
    materiais_especificos.append({
        "material_id": new_id(), "categoria_id": cat_id, "cooperativa_id": coop["cooperativa_id"],
        "imagem_url": None, "preco_sugerido": preco, "esta_disponivel": True,
    })

todos_materiais_novos = materiais_genericos + materiais_especificos
bulk_insert(cur, "materiais",
            ["material_id", "categoria_id", "cooperativa_id", "imagem_url", "preco_sugerido",
             "esta_disponivel"], todos_materiais_novos)

materiais_genericos_ids = [m["material_id"] for m in materiais_genericos]
material_by_id = {m["material_id"]: m for m in todos_materiais_novos}
especifico_by_coop = {m["cooperativa_id"]: m["material_id"] for m in materiais_especificos}

print("categorias novas:", len(categorias_novas_rows), "| materiais genéricos:",
      len(materiais_genericos), "| materiais específicos:", len(materiais_especificos))


categorias novas: 21 | materiais genéricos: 60 | materiais específicos: 10


## 11. Fase 5 — Usuários e Funcionários (1:1 Usuário↔Funcionário; N:1 Funcionário→Cooperativa)

Cada `funcionario` referencia um `usuario`, uma `cooperativa` (nula apenas
para admins de plataforma) e um `cargo`. Garantimos **pelo menos 1 Gestor
por cooperativa**, pois `equipes.gestor_id` e `rateios.gestor_id` exigem um
funcionário gestor válido mais adiante.

`esta_ativo` e `esta_afastado` são colunas geradas (`GENERATED ALWAYS AS
... STORED`) — por isso **não** entram no `INSERT`.


In [11]:
admin_site_mocks = [
    ("Rafael Takematsu", "rafael.takematsu@renovai.com", "123.456.789-10"),
    ("Iago Balbino", "iago.balbino@renovai.com", "123.456.789-11"),
    ("Corbucci", "corbucci.eats@renovai.com", "123.456.789-12"),
    ("Gabinha", "gabriel.andozia@renovai.com", "123.456.789-13"),
    ("Caio Caio", "caio.caio@renovai.com", "123.456.789-14"),
    ("Kauã Ribeiro", "kaua.ribeiro@renovai.com", "123.456.789-15"),
]

usuarios = []
funcionarios = []
for nome, email, cpf in admin_site_mocks:
    usuario_id = new_id()
    usuarios.append({
        "usuario_id": usuario_id, "nome": nome, "email": email, "cpf": cpf,
        "senha": None, "senha_hash": gerar_token(), "imagem_url": None,
        "data_nascimento": fake.date_of_birth(minimum_age=22, maximum_age=60),
        "esta_ativo": True,
    })
    funcionarios.append({
        "funcionario_id": new_id(), "usuario_id": usuario_id, "cooperativa_id": None,
        "cargo_id": cargos["ADMIN_SITE"], "data_admissao": today,
        "data_desligamento": None, "status_funcionario": "ATIVO",
        "_cargo": "ADMIN_SITE",
    })

bulk_insert(cur, "usuarios",
            ["usuario_id", "nome", "email", "cpf", "senha", "senha_hash", "imagem_url",
             "data_nascimento", "esta_ativo"], usuarios)
bulk_insert(cur, "funcionarios",
            ["funcionario_id", "usuario_id", "cooperativa_id", "cargo_id", "data_admissao",
             "data_desligamento", "status_funcionario"], funcionarios)

print("ADMIN_SITE inseridos:", len(admin_site_mocks))


ADMIN_SITE inseridos: 6


In [12]:
funcionarios_by_coop = {c["cooperativa_id"]: [] for c in cooperativas}
gestor_by_coop = {}
usuarios_volume = []
funcionarios_volume = []

for coop in cooperativas:
    coop_id = coop["cooperativa_id"]
    cargos_desta_coop = (["Gestor"] * 1 + ["Admin"] * 2 + ["motorista"] * 2 +
                          ["Cooperado"] * (N_USUARIOS_POR_COOP - 5))
    for cargo_nome in cargos_desta_coop:
        uid = new_id()
        email = fake.unique.email()
        usuario = {
            "usuario_id": uid, "nome": fake.name(), "email": email,
            "cpf": fake.unique.cpf(), "senha": None,
            "senha_hash": gerar_token(),
            "imagem_url": None, "data_nascimento": fake.date_of_birth(minimum_age=18, maximum_age=70),
            "esta_ativo": random.random() < 0.97,
        }
        usuarios_volume.append(usuario)
        usuarios.append(usuario)

        status_func = np.random.choice(["ATIVO", "AFASTADO", "INATIVO"], p=[0.88, 0.05, 0.07])
        data_admissao = fake.date_between(start_date="-3y", end_date="-30d")
        data_desligamento = None
        if status_func == "INATIVO":
            data_desligamento = data_admissao + datetime.timedelta(days=random.randint(60, 500))
            if data_desligamento >= today:
                data_desligamento = today - datetime.timedelta(days=1)
        func = {
            "funcionario_id": new_id(), "usuario_id": uid, "cooperativa_id": coop_id,
            "cargo_id": cargos[cargo_nome], "data_admissao": data_admissao,
            "data_desligamento": data_desligamento, "status_funcionario": str(status_func),
            "_cargo": cargo_nome,
        }
        funcionarios_volume.append(func)
        funcionarios.append(func)
        funcionarios_by_coop[coop_id].append(func)
        if cargo_nome == "Gestor" and coop_id not in gestor_by_coop:
            gestor_by_coop[coop_id] = func

for _ in range(N_ADMINS_PLATAFORMA):
    uid = new_id()
    usuario = {
        "usuario_id": uid, "nome": fake.name(), "email": fake.unique.email(),
        "cpf": fake.unique.cpf(), "senha": None, "senha_hash": gerar_token(),
        "imagem_url": None, "data_nascimento": fake.date_of_birth(minimum_age=22, maximum_age=60),
        "esta_ativo": True,
    }
    usuarios_volume.append(usuario)
    usuarios.append(usuario)
    func = {
        "funcionario_id": new_id(), "usuario_id": uid, "cooperativa_id": None,
        "cargo_id": cargos["Admin"], "data_admissao": fake.date_between(start_date="-3y", end_date="-30d"),
        "data_desligamento": None, "status_funcionario": "ATIVO", "_cargo": "Admin",
    }
    funcionarios_volume.append(func)
    funcionarios.append(func)

bulk_insert(cur, "usuarios",
            ["usuario_id", "nome", "email", "cpf", "senha", "senha_hash", "imagem_url",
             "data_nascimento", "esta_ativo"], usuarios_volume)
bulk_insert(cur, "funcionarios",
            ["funcionario_id", "usuario_id", "cooperativa_id", "cargo_id", "data_admissao",
             "data_desligamento", "status_funcionario"], funcionarios_volume)

print("usuarios de volume:", len(usuarios_volume), "| funcionarios de volume:", len(funcionarios_volume))
print("totais com ADMIN_SITE:", len(usuarios), "usuarios |", len(funcionarios), "funcionarios")


usuarios de volume: 400 | funcionarios de volume: 400
totais com ADMIN_SITE: 406 usuarios | 406 funcionarios


## 12. Fase 6 — Telefones

`telefones` tem `CHECK` de dono exclusivo: pertence a um `perfil` **ou** a
um `usuario`, nunca os dois.


In [13]:
telefones = []
for perfil in perfis:
    telefones.append({
        "telefone_id": new_id(), "perfil_id": perfil["perfil_id"], "usuario_id": None,
        "ddd": uniq_ddd(), "telefone": uniq_telefone_local(),
    })
for func in funcionarios:
    n_tel = np.random.choice([1, 2], p=[0.8, 0.2])
    for _ in range(int(n_tel)):
        telefones.append({
            "telefone_id": new_id(), "perfil_id": None, "usuario_id": func["usuario_id"],
            "ddd": uniq_ddd(), "telefone": uniq_telefone_local(),
        })
bulk_insert(cur, "telefones", ["telefone_id", "perfil_id", "usuario_id", "ddd", "telefone"], telefones)
print("telefones:", len(telefones))


telefones: 545


## 13. Fase 7 — Equipes, Equipes×Cooperados (N:N), Rotas e Rotas×Endereços

`equipes_cooperados` é a tabela N:N entre equipes e cooperados
(`funcionarios` com cargo Cooperado). A trigger
`trg_validar_cooperativa_funcionario` garante que o cooperado pertence à
**mesma cooperativa** do gestor da equipe — por isso escolhemos cooperados
sempre dentro de `funcionarios_by_coop[coop_id]`.


In [14]:
equipes = []
equipes_by_coop = {c["cooperativa_id"]: [] for c in cooperativas}
for coop in cooperativas:
    coop_id = coop["cooperativa_id"]
    gestor = gestor_by_coop.get(coop_id)
    if gestor is None:
        continue
    for letra in ["A", "B"]:
        equipe = {"equipe_id": new_id(), "gestor_id": gestor["funcionario_id"],
                  "nome": f"Equipe {letra} - {coop['nome']}", "descricao": "Equipe de triagem e coleta"}
        equipes.append(equipe)
        equipes_by_coop[coop_id].append(equipe)
bulk_insert(cur, "equipes", ["equipe_id", "gestor_id", "nome", "descricao"], equipes)

equipes_cooperados = []
for coop in cooperativas:
    coop_id = coop["cooperativa_id"]
    eqs = equipes_by_coop[coop_id]
    if not eqs:
        continue
    cooperados = [f for f in funcionarios_by_coop[coop_id] if f["_cargo"] == "Cooperado"]
    for idx, coop_func in enumerate(cooperados):
        eq = eqs[idx % len(eqs)]
        equipes_cooperados.append({"equipe_cooperado_id": new_id(), "equipe_id": eq["equipe_id"],
                                    "cooperado_id": coop_func["funcionario_id"]})
bulk_insert(cur, "equipes_cooperados", ["equipe_cooperado_id", "equipe_id", "cooperado_id"],
            equipes_cooperados)

rotas = []
rotas_by_coop = {c["cooperativa_id"]: [] for c in cooperativas}
for coop in cooperativas:
    coop_id = coop["cooperativa_id"]
    for letra in ["Norte", "Sul"]:
        rota = {"rota_id": new_id(), "cooperativa_id": coop_id, "nome": f"Rota {letra} - {coop['nome']}"}
        rotas.append(rota)
        rotas_by_coop[coop_id].append(rota)
bulk_insert(cur, "rotas", ["rota_id", "cooperativa_id", "nome"], rotas)

rotas_enderecos = []
for rota in rotas:
    n_pontos = random.randint(2, 6)
    amostra = random.sample(enderecos_coleta, min(n_pontos, len(enderecos_coleta)))
    for ordem, end in enumerate(amostra, start=1):
        rotas_enderecos.append({
            "rota_endereco_id": new_id(), "rota_id": rota["rota_id"], "endereco_id": end["endereco_id"],
            "nome_local": f"Ponto {fake.bairro()}", "tipo_local": "Ponto de Coleta", "ordem": ordem,
        })
bulk_insert(cur, "rotas_enderecos",
            ["rota_endereco_id", "rota_id", "endereco_id", "nome_local", "tipo_local", "ordem"],
            rotas_enderecos)

print("equipes:", len(equipes), "| equipes_cooperados:", len(equipes_cooperados),
      "| rotas:", len(rotas), "| rotas_enderecos:", len(rotas_enderecos))


equipes: 36 | equipes_cooperados: 306 | rotas: 36 | rotas_enderecos: 141


## 14. Fase 8 — Estoques (carga base de ~5.000 kg por material)

`estoques` é `INSERT` direto (é o saldo inicial/de abertura, diferente de
`movimentacoes_estoques`, que só é gerada por procedure). Criamos um saldo
base de **5.000 kg com variação** (`numpy`) para cada combinação
cooperativa × material genérico, e também para os materiais específicos em
sua própria cooperativa.

O dicionário Python `stock[(cooperativa_id, material_id)]` passa a
**espelhar o saldo real do banco** a partir daqui — ele é atualizado em
Python toda vez que uma ação (triagem, aceite de pedido, fechamento de
negociação) também altera o saldo no banco via trigger
`fn_sincronizar_saldo_estoque`. É esse espelho que usamos para decidir,
**antes de cada operação**, se uma cooperativa tem estoque suficiente —
exatamente a regra pedida: *pedidos não podem ser direcionados a
cooperativas sem estoque suficiente do material*.

> **Proteção contra duplicidade:** `estoques` tem
> `UNIQUE(cooperativa_id, material_id)` — cada cooperativa deve ter **um único
> `estoque_id` por material**, nunca dois. O `INSERT` abaixo usa
> `ON CONFLICT (cooperativa_id, material_id) DO NOTHING`, o mesmo padrão que
> `sp_registrar_movimentacao_triagem` já usa no banco para essa tabela. Isso
> torna a carga seguro para **reexecutar**: se o notebook rodar de novo no
> mesmo banco, a segunda tentativa de criar a mesma combinação
> cooperativa+material é simplesmente ignorada (mantendo o `estoque_id` e o
> saldo já existentes) em vez de gerar uma segunda linha ou quebrar com erro
> de `unique_violation`.


In [15]:
estoques_rows = []
stock = {}  # (cooperativa_id, material_id) -> quantidade_kg (espelha o saldo real do banco)

for coop in cooperativas:
    coop_id = coop["cooperativa_id"]
    for mat_id in materiais_genericos_ids:
        qtd = round(float(ESTOQUE_BASE_KG + np.random.uniform(-1500, 3500)), 3)
        qtd = max(qtd, 300.0)
        estoques_rows.append({"estoque_id": new_id(), "cooperativa_id": coop_id,
                               "material_id": mat_id, "quantidade_kg": qtd})
        stock[(coop_id, mat_id)] = qtd

for mat in materiais_especificos:
    coop_id = mat["cooperativa_id"]
    qtd = round(float(ESTOQUE_BASE_KG + np.random.uniform(-1500, 3500)), 3)
    qtd = max(qtd, 300.0)
    estoques_rows.append({"estoque_id": new_id(), "cooperativa_id": coop_id,
                           "material_id": mat["material_id"], "quantidade_kg": qtd})
    stock[(coop_id, mat["material_id"])] = qtd

# ON CONFLICT garante 1 único estoque_id por (cooperativa_id, material_id):
# reexecutar esta célula/notebook no mesmo banco não cria linha duplicada.
bulk_insert(cur, "estoques", ["estoque_id", "cooperativa_id", "material_id", "quantidade_kg"],
            estoques_rows, on_conflict="ON CONFLICT (cooperativa_id, material_id) DO NOTHING")

# Se a carga já existia (reexecução), o dicionário `stock` acima ficaria com os
# valores novos sorteados agora, não os que realmente estão no banco (que foram
# preservados pelo DO NOTHING) — buscamos o saldo real para manter os dois em
# sincronia. `uuid.UUID(str(...))` normaliza o tipo devolvido pelo driver (com
# psycopg2.extras.register_uuid() já deveria vir como uuid.UUID, mas o cast
# explícito garante que a chave bate com a usada em `new_id()` mesmo se algum
# dia isso mudar) — sem essa normalização, uma chave seria uuid.UUID e a outra
# string, e o dicionário acabaria com o DOBRO de entradas em vez de substituir.
stock = {}
cur.execute("SELECT cooperativa_id, material_id, quantidade_kg FROM estoques")
for coop_id, mat_id, qtd_real in cur.fetchall():
    stock[(uuid.UUID(str(coop_id)), uuid.UUID(str(mat_id)))] = float(qtd_real)

print("estoques (carga base gerada nesta execução):", len(estoques_rows),
      "| linhas totais em `estoques` no banco:", len(stock))


estoques (carga base gerada nesta execução): 1090 | linhas totais em `estoques` no banco: 1090


## 15. Fase 9 — Despesas e Lançamentos de Despesas

Necessárias para que `fn_calcular_total_liquido` (usada pelo rateio
automático, fase 13) tenha despesas reais a descontar da receita bruta.


In [16]:
DESPESAS_TEMPLATE = [
    ("Aluguel do galpão", "FIXA", (1200, 4000)),
    ("Combustível da frota", "VARIAVEL", (500, 2500)),
    ("Manutenção de equipamentos", "VARIAVEL", (200, 1800)),
]
despesas = []
despesas_by_coop = {c["cooperativa_id"]: [] for c in cooperativas}
for coop in cooperativas:
    coop_id = coop["cooperativa_id"]
    for nome, tipo, faixa in DESPESAS_TEMPLATE:
        d = {"despesa_id": new_id(), "cooperativa_id": coop_id, "nome": nome,
             "tipo_despesa": tipo, "_faixa": faixa}
        despesas.append(d)
        despesas_by_coop[coop_id].append(d)
bulk_insert(cur, "despesas", ["despesa_id", "cooperativa_id", "nome", "tipo_despesa"], despesas)

# últimos 3 meses fechados (mes_referencia é truncado para o 1º dia do mês pela
# trigger trg_normalizar_mes_referencia_lancamentos; passamos qualquer dia do mês)
meses_referencia = []
mes_cursor = today.replace(day=1)
for _ in range(3):
    meses_referencia.append(mes_cursor)
    mes_cursor = (mes_cursor - datetime.timedelta(days=1)).replace(day=1)
meses_referencia.reverse()

lancamentos = []
for d in despesas:
    lo, hi = d["_faixa"]
    for mes in meses_referencia:
        valor = round(float(np.random.uniform(lo, hi)), 2)
        lancamentos.append({"lancamento_id": new_id(), "despesa_id": d["despesa_id"],
                             "valor": valor, "mes_referencia": mes})
bulk_insert(cur, "lancamentos_despesas", ["lancamento_id", "despesa_id", "valor", "mes_referencia"],
            lancamentos)
print("despesas:", len(despesas), "| lancamentos_despesas:", len(lancamentos))
print("meses de referência:", meses_referencia)


despesas: 54 | lancamentos_despesas: 162
meses de referência: [datetime.date(2026, 7, 1), datetime.date(2026, 8, 1), datetime.date(2026, 9, 1)]


## 16. Fase 10a — Pedidos e Itens de Pedido

`pedido_itens.quantidade_kg` pode ser `NULL` (o `CHECK` permite) — isso
representa um item cujo peso ainda **não foi definido**, forçando o fluxo
de negociação (a procedure `sp_aceitar_pedido_cooperativa` rejeita pedidos
com item sem peso). Reservamos ~25% dos pedidos para esse cenário.

Também injetamos, em ~8% dos pedidos "diretos", um item com quantidade bem
acima da carga base (3.000–9.000 kg) — propositalmente para exercitar o
cenário em que **nenhuma cooperativa tem estoque suficiente** (fase 10b).

Note que só usamos `materiais_genericos_ids` aqui: a trigger
`fn_validar_material_item` proíbe que um item de pedido referencie um
material específico de cooperativa.


In [17]:
status_pedido_aberto = status[("PEDIDO", "Aberto")]
status_pedido_aceito = status[("PEDIDO", "Aceito")]
status_pedido_recusado = status[("PEDIDO", "Recusado")]
status_pedido_negociacao = status[("PEDIDO", "Em negociação")]
status_negociacao_andamento = status[("NEGOCIACAO", "Em andamento")]

pedidos = []
pedido_itens_all = []
pedidos_plan = []  # metadados em memória usados na fase 10b

for _ in range(N_PEDIDOS):
    pedido_id = new_id()
    empresa = random.choice(empresas)
    data_pedido = fake.date_between(start_date="-150d", end_date="-5d")
    pedidos.append({
        "pedido_id": pedido_id, "empresa_id": empresa["empresa_id"],
        "observacao": None if random.random() < 0.4 else fake.sentence(),
        "data_pedido": data_pedido, "data_conclusao": None,
    })

    n_itens = random.randint(1, 4)
    materiais_pedido = random.sample(materiais_genericos_ids, n_itens)
    precisa_negociar = random.random() < 0.25
    tem_item_grande = (not precisa_negociar) and random.random() < 0.08
    idx_grande = random.randrange(n_itens) if tem_item_grande else -1
    idx_sem_peso = random.randrange(n_itens) if precisa_negociar else -1

    itens_pedido = []
    for idx, mat_id in enumerate(materiais_pedido):
        mat = material_by_id[mat_id]
        preco = round(float(mat["preco_sugerido"]) * float(np.random.uniform(0.85, 1.15)), 2)
        if idx == idx_sem_peso:
            qtd = None
        elif idx == idx_grande:
            qtd = round(float(np.random.uniform(3000, 9000)), 3)
        else:
            qtd = round(float(np.random.uniform(20, 400)), 3)
        itens_pedido.append({"item_id": new_id(), "pedido_id": pedido_id, "material_id": mat_id,
                              "quantidade_kg": qtd, "preco_unitario": preco})
    pedido_itens_all.extend(itens_pedido)

    pedidos_plan.append({"pedido_id": pedido_id, "empresa_id": empresa["empresa_id"],
                          "data_pedido": data_pedido, "itens": itens_pedido,
                          "precisa_negociar": precisa_negociar})

bulk_insert(cur, "pedidos", ["pedido_id", "empresa_id", "observacao", "data_pedido", "data_conclusao"],
            pedidos)
bulk_insert(cur, "pedido_itens",
            ["item_id", "pedido_id", "material_id", "quantidade_kg", "preco_unitario"],
            pedido_itens_all)
print("pedidos:", len(pedidos), "| pedido_itens:", len(pedido_itens_all))


pedidos: 260 | pedido_itens: 622


## 17. Fase 10b — Vínculo Pedido×Cooperativa (N:N), aceite direto e negociação

Esta é a fase que aplica a regra central do enunciado. Para cada pedido:

- **Sem itens em aberto (peso definido em todos):** calculamos as
  cooperativas *elegíveis* — aquelas cujo saldo em `stock` comporta **todos**
  os itens do pedido. Só essas entram em `pedidos_cooperativas`. Se **nenhuma**
  cooperativa for elegível (pedidos com item de quantidade muito alta), o
  pedido **não recebe nenhum vínculo** — fica aberto sem cooperativa capaz de
  atendê-lo, e isso é contado e exibido no resumo abaixo.
  Para uma parte dos vínculos criados, chamamos
  `CALL sp_aceitar_pedido_cooperativa(...)` (a própria procedure reconfere o
  estoque via `fn_verificar_disponibilidade_estoque` antes de gerar as
  movimentações de saída); outra parte é marcada como `Recusado`
  (`UPDATE` direto) e o restante permanece `Aberto`.
- **Com item(ns) sem peso definido:** vai direto para negociação — criamos
  `pedidos_cooperativas` com status `Em negociação`, e uma `negociacoes`
  (com `negociacao_itens` fechando os pesos/preços e `negociacao_mensagens`
  simulando a conversa). Ao final, chamamos
  `CALL sp_fechar_negociacao(negociacao_id, aceito)` — `aceito=True` apenas
  quando o estoque (nosso espelho `stock`) realmente comporta os itens
  negociados, senão a negociação é fechada como recusada.

Em ambos os casos, a cada aceite bem-sucedido subtraímos do dicionário
`stock` local exatamente o que a trigger `fn_sincronizar_saldo_estoque`
subtrairá no banco — mantendo os dois em sincronia para as decisões
seguintes.


In [18]:
pedidos_update_conclusao = {}
concluidos_empresa_coop = []  # (pedido_id, empresa_id, cooperativa_id) fechados com sucesso
n_sem_cooperativa_disponivel = 0
n_pedidos_cooperativas = 0
n_negociacoes = 0

for plano in pedidos_plan:
    pedido_id = plano["pedido_id"]
    empresa_id = plano["empresa_id"]
    data_pedido = plano["data_pedido"]
    itens_pedido = plano["itens"]
    ja_concluido = False

    if plano["precisa_negociar"]:
        candidatas = [c for c in cooperativas
                      if any((c["cooperativa_id"], it["material_id"]) in stock for it in itens_pedido)]
        escolhidas = random.sample(candidatas, min(1, len(candidatas))) if candidatas else []
        for coop in escolhidas:
            coop_id = coop["cooperativa_id"]
            pc_id = new_id()
            bulk_insert(cur, "pedidos_cooperativas",
                        ["pedido_cooperativa_id", "pedido_id", "cooperativa_id", "status_id"],
                        [{"pedido_cooperativa_id": pc_id, "pedido_id": pedido_id,
                          "cooperativa_id": coop_id, "status_id": status_pedido_negociacao}])
            n_pedidos_cooperativas += 1

            negociacao_id = new_id()
            data_inicio = data_pedido + datetime.timedelta(days=random.randint(0, 3))
            bulk_insert(cur, "negociacoes",
                        ["negociacao_id", "pedido_id", "cooperativa_id", "empresa_id", "status_id",
                         "valor_total", "data_inicio"],
                        [{"negociacao_id": negociacao_id, "pedido_id": pedido_id,
                          "cooperativa_id": coop_id, "empresa_id": empresa_id,
                          "status_id": status_negociacao_andamento, "valor_total": None,
                          "data_inicio": data_inicio}])
            n_negociacoes += 1

            neg_itens = []
            ok_estoque = True
            for it in itens_pedido:
                qtd_final = it["quantidade_kg"]
                if qtd_final is None:
                    qtd_final = round(float(np.random.uniform(20, 400)), 3)
                preco_final = round(it["preco_unitario"] * float(np.random.uniform(0.95, 1.05)), 2)
                neg_itens.append({"negociacao_item_id": new_id(), "negociacao_id": negociacao_id,
                                   "material_id": it["material_id"], "quantidade_kg": qtd_final,
                                   "preco_unitario": preco_final})
                if stock.get((coop_id, it["material_id"]), 0.0) < qtd_final:
                    ok_estoque = False
            bulk_insert(cur, "negociacao_itens",
                        ["negociacao_item_id", "negociacao_id", "material_id", "quantidade_kg",
                         "preco_unitario"], neg_itens)

            perfil_empresa = perfil_by_empresa[empresa_id]
            perfil_coop = perfil_by_cooperativa[coop_id]
            n_msgs = random.randint(2, 5)
            remetentes = [perfil_empresa, perfil_coop]
            msgs = []
            for m in range(n_msgs):
                msgs.append({"mensagem_id": new_id(), "negociacao_id": negociacao_id,
                              "remetente_id": remetentes[m % 2], "mensagem": fake.sentence(),
                              "tipo_mensagem": "CONTRAPROPOSTA" if m == 1 else "TEXTO",
                              "data_envio": data_inicio + datetime.timedelta(hours=m * 3)})
            bulk_insert(cur, "negociacao_mensagens",
                        ["mensagem_id", "negociacao_id", "remetente_id", "mensagem", "tipo_mensagem",
                         "data_envio"], msgs)

            aceitar = ok_estoque and random.random() < 0.7
            valor_total = round(sum(ni["quantidade_kg"] * ni["preco_unitario"] for ni in neg_itens), 2)
            update_one(cur, "negociacoes", {"valor_total": valor_total}, "negociacao_id", negociacao_id)
            call_proc(cur, "sp_fechar_negociacao", [negociacao_id, bool(aceitar)])
            if aceitar:
                for ni in neg_itens:
                    stock[(coop_id, ni["material_id"])] -= ni["quantidade_kg"]
                conclusao = data_inicio + datetime.timedelta(days=random.randint(1, 5))
                pedidos_update_conclusao[pedido_id] = conclusao
                concluidos_empresa_coop.append((pedido_id, empresa_id, coop_id))
    else:
        elegiveis = [
            c for c in cooperativas
            if all(stock.get((c["cooperativa_id"], it["material_id"]), 0.0) >= it["quantidade_kg"]
                   for it in itens_pedido)
        ]
        if not elegiveis:
            n_sem_cooperativa_disponivel += 1  # regra respeitada: nenhum vínculo é criado
        else:
            escolhidas = random.sample(elegiveis, min(2, len(elegiveis)))
            for coop in escolhidas:
                coop_id = coop["cooperativa_id"]
                pc_id = new_id()
                bulk_insert(cur, "pedidos_cooperativas",
                            ["pedido_cooperativa_id", "pedido_id", "cooperativa_id", "status_id"],
                            [{"pedido_cooperativa_id": pc_id, "pedido_id": pedido_id,
                              "cooperativa_id": coop_id, "status_id": status_pedido_aberto}])
                n_pedidos_cooperativas += 1

                sorteio = random.random()
                if (not ja_concluido) and sorteio < 0.65:
                    ainda_ok = all(stock.get((coop_id, it["material_id"]), 0.0) >= it["quantidade_kg"]
                                   for it in itens_pedido)
                    if ainda_ok:
                        call_proc(cur, "sp_aceitar_pedido_cooperativa", [pc_id, status_pedido_aceito])
                        for it in itens_pedido:
                            stock[(coop_id, it["material_id"])] -= it["quantidade_kg"]
                        conclusao = data_pedido + datetime.timedelta(days=random.randint(1, 7))
                        pedidos_update_conclusao[pedido_id] = conclusao
                        concluidos_empresa_coop.append((pedido_id, empresa_id, coop_id))
                        ja_concluido = True
                elif sorteio < 0.80:
                    update_one(cur, "pedidos_cooperativas", {"status_id": status_pedido_recusado},
                               "pedido_cooperativa_id", pc_id)
                # senão: permanece 'Aberto'

for pedido_id, data_conclusao in pedidos_update_conclusao.items():
    update_one(cur, "pedidos", {"data_conclusao": data_conclusao}, "pedido_id", pedido_id)

print("pedidos_cooperativas:", n_pedidos_cooperativas, "| negociacoes:", n_negociacoes,
      "| pedidos concluídos:", len(concluidos_empresa_coop))
print("pedidos SEM cooperativa com estoque suficiente (regra respeitada, nenhum vínculo criado):",
      n_sem_cooperativa_disponivel)


pedidos_cooperativas: 439 | negociacoes: 75 | pedidos concluídos: 206
pedidos SEM cooperativa com estoque suficiente (regra respeitada, nenhum vínculo criado): 3


## 18. Fase 11a — Coletas

`coletas` herda de `eventos_operacionais` (`INHERITS`), mas a PK
(`evento_id`) e o `DEFAULT gen_random_uuid()` são herdados normalmente pelo
PostgreSQL — por isso geramos o UUID em Python (para poder referenciá-lo
depois nas triagens) e inserimos normalmente.


In [19]:
status_coleta_coletado = status[("COLETA", "Coletado")]
status_coleta_agendada = status[("COLETA", "Agendada")]
status_coleta_pendente = status[("COLETA", "Pendente")]
status_triagem_concluido = status[("TRIAGEM", "Concluído")]
status_triagem_em_triagem = status[("TRIAGEM", "Em triagem")]

funcionarios_coletores_by_coop = {
    coop_id: [f for f in flist if f["_cargo"] in ("Cooperado", "motorista")
              and f["status_funcionario"] == "ATIVO"]
    for coop_id, flist in funcionarios_by_coop.items()
}

coletas = []
coletas_plan = []
for _ in range(N_COLETAS):
    coop = random.choice(cooperativas)
    coop_id = coop["cooperativa_id"]
    coletores = funcionarios_coletores_by_coop.get(coop_id) or []
    if not coletores:
        continue
    cooperado = random.choice(coletores)
    tipo = random.choice(["EXTERNA", "EXTERNA", "ENTREGA"])
    rota = random.choice(rotas_by_coop[coop_id]) if (tipo == "EXTERNA" and rotas_by_coop[coop_id]) else None
    status_sorteio = random.random()
    if status_sorteio < 0.75:
        status_id = status_coleta_coletado
    elif status_sorteio < 0.90:
        status_id = status_coleta_agendada
    else:
        status_id = status_coleta_pendente
    evento_id = new_id()
    quantidade = round(float(np.random.uniform(15, 320)), 3)
    coletas.append({
        "evento_id": evento_id, "data_evento": fake.date_between(start_date="-120d", end_date="-1d"),
        "status_id": status_id, "cooperado_id": cooperado["funcionario_id"],
        "rota_id": rota["rota_id"] if rota else None, "imagem_url": None,
        "tipo_coleta": tipo, "quantidade_kg": quantidade,
    })
    coletas_plan.append({"evento_id": evento_id, "coop_id": coop_id, "quantidade": quantidade,
                          "status_id": status_id})

bulk_insert(cur, "coletas",
            ["evento_id", "data_evento", "status_id", "cooperado_id", "rota_id", "imagem_url",
             "tipo_coleta", "quantidade_kg"], coletas)
print("coletas:", len(coletas))


coletas: 450


## 19. Fase 11b — Triagens (**apenas via procedure**)

`triagens` e as `movimentacoes_estoques` de entrada correspondentes **não
recebem `INSERT` direto** — são geradas exclusivamente por
`CALL sp_registrar_movimentacao_triagem(...)`, que:

1. garante a existência do registro de `estoques` daquela
   cooperativa×material (`ON CONFLICT DO NOTHING`);
2. insere a `triagem`;
3. insere a `movimentacao_estoque` de entrada (o que, via trigger
   `fn_sincronizar_saldo_estoque`, soma ao saldo de `estoques`).

Para cada coleta já `Coletado`, dividimos o peso coletado em 1–2 triagens
(equipe da mesma cooperativa do cooperado, material genérico — ou, com baixa
probabilidade, o material específico daquela cooperativa, quando existir).
Deixamos `p_quantidade_rejeito_kg=None` para que o próprio banco calcule o
rejeito via `fn_calcular_quantidade_rejeito_kg`.


In [20]:
n_triagens_chamadas = 0
for plano in coletas_plan:
    if plano["status_id"] != status_coleta_coletado:
        continue
    coop_id = plano["coop_id"]
    equipes_coop = equipes_by_coop.get(coop_id) or []
    if not equipes_coop:
        continue
    total = plano["quantidade"]
    n_splits = 1 if total < 60 or random.random() < 0.5 else 2
    restante = total
    for i in range(n_splits):
        if restante <= 5:
            break
        if i == n_splits - 1:
            porcao = restante
        else:
            porcao = round(restante * float(np.random.uniform(0.3, 0.6)), 3)
        restante = round(restante - porcao, 3)
        equipe = random.choice(equipes_coop)
        if coop_id in especifico_by_coop and random.random() < 0.08:
            material_id = especifico_by_coop[coop_id]
        else:
            material_id = random.choice(materiais_genericos_ids)
        status_triagem = status_triagem_concluido if random.random() < 0.85 else status_triagem_em_triagem
        call_proc(cur, "sp_registrar_movimentacao_triagem",
                  [equipe["equipe_id"], plano["evento_id"], material_id, status_triagem, porcao, None])
        n_triagens_chamadas += 1
        stock[(coop_id, material_id)] = stock.get((coop_id, material_id), 0.0) + porcao

print("chamadas CALL sp_registrar_movimentacao_triagem:", n_triagens_chamadas)


chamadas CALL sp_registrar_movimentacao_triagem: 493


## 20. Fase 12 — Avaliações

`avaliacoes` liga `perfis` (avaliador/avaliado), nunca `funcionarios` — e o
`CHECK chk_avaliacoes_perfis_diferentes` exige `avaliador_id <>
avaliado_id`. Para cada pedido concluído (aceito direto ou via negociação),
geramos a avaliação da empresa sobre a cooperativa e vice-versa.


In [21]:
avaliacoes = []
amostra_concluidos = (random.sample(concluidos_empresa_coop, int(len(concluidos_empresa_coop) * 0.7))
                       if concluidos_empresa_coop else [])
for pedido_id, empresa_id, coop_id in amostra_concluidos:
    perfil_empresa = perfil_by_empresa[empresa_id]
    perfil_coop = perfil_by_cooperativa[coop_id]
    data_base = fake.date_between(start_date="-90d", end_date="-1d")
    nota_empresa = int(np.random.choice([1, 2, 3, 4, 5], p=[0.03, 0.05, 0.17, 0.35, 0.40]))
    nota_coop = int(np.random.choice([1, 2, 3, 4, 5], p=[0.02, 0.04, 0.14, 0.35, 0.45]))
    avaliacoes.append({"avaliacao_id": new_id(), "avaliador_id": perfil_empresa,
                        "avaliado_id": perfil_coop, "pedido_id": pedido_id, "nota": nota_empresa,
                        "comentario": fake.sentence(), "data_avaliacao": data_base})
    avaliacoes.append({"avaliacao_id": new_id(), "avaliador_id": perfil_coop,
                        "avaliado_id": perfil_empresa, "pedido_id": pedido_id, "nota": nota_coop,
                        "comentario": fake.sentence(), "data_avaliacao": data_base})
bulk_insert(cur, "avaliacoes",
            ["avaliacao_id", "avaliador_id", "avaliado_id", "pedido_id", "nota", "comentario",
             "data_avaliacao"], avaliacoes)
print("avaliacoes:", len(avaliacoes))


avaliacoes: 288


## 21. Fase 13 — Rateios (**apenas via procedure**)

`rateios` e `rateios_funcionarios` **não recebem `INSERT` direto** — só são
criados por `CALL sp_fechar_rateio_mensal(cooperativa_id, gestor_id,
mes_referencia)`, que calcula o valor líquido do mês
(`fn_calcular_rateio_automatico` → `fn_calcular_total_liquido`, usando os
`pedidos` concluídos e as `despesas`/`lancamentos_despesas` já carregados) e
distribui entre os funcionários ativos da cooperativa.

Chamamos a procedure para cada cooperativa × cada um dos 3 últimos meses
fechados.


In [22]:
n_rateios_chamados = 0
for coop in cooperativas:
    coop_id = coop["cooperativa_id"]
    gestor = gestor_by_coop.get(coop_id)
    if gestor is None:
        continue
    for mes in meses_referencia:
        mes_ts = datetime.datetime.combine(mes, datetime.time.min)
        call_proc(cur, "sp_fechar_rateio_mensal", [coop_id, gestor["funcionario_id"], mes_ts])
        n_rateios_chamados += 1
print("chamadas CALL sp_fechar_rateio_mensal:", n_rateios_chamados)


chamadas CALL sp_fechar_rateio_mensal: 54


## 22. Fase 14 — Interesses e Favoritos de Empresas (N:N)

Duas tabelas associativas simples: `empresa_materiais_interesses`
(empresa × categoria de material) e `empresa_cooperativas_favoritas`
(empresa × cooperativa), ambas com restrição `UNIQUE` de par — controlada em
Python com um `set()` para não gerar duplicidade.


In [23]:
categoria_ids_all = list(categoria_by_nome.values()) + list(novas_macro.values()) + \
                    [cid for cid, _n, _m in leaf_categorias]

interesses = []
pares_interesse = set()
for emp in empresas:
    n = random.randint(1, 4)
    for cat_id in random.sample(categoria_ids_all, min(n, len(categoria_ids_all))):
        chave = (emp["empresa_id"], cat_id)
        if chave in pares_interesse:
            continue
        pares_interesse.add(chave)
        interesses.append({"empresa_material_id": new_id(), "empresa_id": emp["empresa_id"],
                            "categoria_id": cat_id})
bulk_insert(cur, "empresa_materiais_interesses", ["empresa_material_id", "empresa_id", "categoria_id"],
            interesses)

favoritos = []
pares_favorito = set()
for emp in empresas:
    n = random.randint(1, 4)
    for coop in random.sample(cooperativas, min(n, len(cooperativas))):
        chave = (emp["empresa_id"], coop["cooperativa_id"])
        if chave in pares_favorito:
            continue
        pares_favorito.add(chave)
        favoritos.append({"favorito_id": new_id(), "empresa_id": emp["empresa_id"],
                           "cooperativa_id": coop["cooperativa_id"]})
bulk_insert(cur, "empresa_cooperativas_favoritas", ["favorito_id", "empresa_id", "cooperativa_id"],
            favoritos)

print("empresa_materiais_interesses:", len(interesses), "| empresa_cooperativas_favoritas:",
      len(favoritos))


empresa_materiais_interesses: 115 | empresa_cooperativas_favoritas: 112


## 23. Verificação final

Conferimos o total de registros inseridos por tabela e algumas checagens de
sanidade das regras de negócio (nenhum estoque negativo, nenhuma avaliação
com avaliador igual ao avaliado, nenhum material específico usado em
`pedido_itens`).


In [24]:
tabelas_para_contar = [
    "enderecos", "categorias_materiais", "materiais", "cooperativas", "empresas", "perfis",
    "usuarios", "funcionarios", "telefones", "equipes", "equipes_cooperados",
    "pedidos", "pedido_itens", "pedidos_cooperativas", "negociacoes", "negociacao_itens",
    "negociacao_mensagens", "avaliacoes", "coletas", "triagens", "estoques",
    "movimentacoes_estoques", "despesas", "lancamentos_despesas", "rotas", "rotas_enderecos",
    "rateios", "rateios_funcionarios", "empresa_materiais_interesses",
    "empresa_cooperativas_favoritas",
]

total_geral = 0
print(f"{'tabela':38s} {'linhas':>8s}")
for t in tabelas_para_contar:
    cur.execute(f"SELECT count(*) FROM {t}")
    n = cur.fetchone()[0]
    total_geral += n
    print(f"{t:38s} {n:8d}")
print("-" * 48)
print(f"{'TOTAL (sem tabelas de auditoria)':38s} {total_geral:8d}")

cur.execute("SELECT count(*) FROM estoques WHERE quantidade_kg < 0")
print("\nEstoques negativos (deve ser 0):", cur.fetchone()[0])

cur.execute("SELECT count(*) FROM avaliacoes WHERE avaliador_id = avaliado_id")
print("Avaliações com avaliador = avaliado (deve ser 0):", cur.fetchone()[0])

cur.execute(
    "SELECT count(*) FROM pedido_itens pi "
    "JOIN materiais m ON m.material_id = pi.material_id "
    "WHERE m.cooperativa_id IS NOT NULL"
)
print("Itens de pedido com material específico de cooperativa (deve ser 0):", cur.fetchone()[0])


tabela                                   linhas
enderecos                                   500
categorias_materiais                         25
materiais                                    74
cooperativas                                 18
empresas                                     45
perfis                                       63
usuarios                                    406
funcionarios                                406
telefones                                   545
equipes                                      36
equipes_cooperados                          306
pedidos                                     260
pedido_itens                                622
pedidos_cooperativas                        439
negociacoes                                  75
negociacao_itens                            170
negociacao_mensagens                        279
avaliacoes                                  288
coletas                                     450
triagens                                

## 24. Encerrar conexão

In [25]:
cur.close()
conn.close()
print("Conexão encerrada. Carga de volume concluída.")


Conexão encerrada. Carga de volume concluída.
